In [4]:
def main(datasources, start_date, end_date):
    """
    盘口成本趋势承接因子 v3

    传统赛道人工假设：
    聪明钱不一定在次日立刻拉升，可能是在下跌趋势中持续压低盘口成本建仓。
    如果盘口成本重心相对过去几天下移，但当日跌幅相对过去趋势收窄，
    说明更低成本区间出现承接，机构成本可能在下移过程中逐步稳定。
    因子使用 T 日完整盘口快照，只能解释为 T 日收盘后形成的日频信号。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    eval_start = pd.to_datetime(start_date)
    query_start = eval_start - pd.Timedelta(days=20)

    # 只使用平台实测可用的五档盘口字段。滚动窗口只看 T 日及以前的数据。
    # query_start 提供历史缓冲，最终输出再裁回平台请求区间。
    sql = f"""
    WITH cte_base AS (
        SELECT
            date,
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            close,
            pre_close,
            bid_price1, bid_price2, bid_price3, bid_price4, bid_price5,
            ask_price1, ask_price2, ask_price3, ask_price4, ask_price5,
            bid_volume1, bid_volume2, bid_volume3, bid_volume4, bid_volume5,
            ask_volume1, ask_volume2, ask_volume3, ask_volume4, ask_volume5,
            ROW_NUMBER() OVER (
                PARTITION BY instrument, strftime(date, '%Y-%m-%d')
                ORDER BY date
            ) AS rn,
            COUNT(*) OVER (
                PARTITION BY instrument, strftime(date, '%Y-%m-%d')
            ) AS cnt
        FROM {bar1m}
        WHERE bid_price1 > 0
          AND ask_price1 > 0
          AND ask_price1 >= bid_price1
          AND close > 0
          AND pre_close > 0
    ),
    cte_snapshot AS (
        SELECT
            date,
            instrument,
            trading_day,
            rn,
            cnt,
            bid_price1,
            ask_price1,
            close,
            pre_close,
            (bid_price1 + ask_price1) / 2.0 AS mid_price,
            (
                COALESCE(bid_volume1, 0) * 1.0
                + COALESCE(bid_volume2, 0) * 0.5
                + COALESCE(bid_volume3, 0) / 3.0
                + COALESCE(bid_volume4, 0) * 0.25
                + COALESCE(bid_volume5, 0) * 0.2
            ) AS bid_weighted_volume,
            (
                COALESCE(ask_volume1, 0) * 1.0
                + COALESCE(ask_volume2, 0) * 0.5
                + COALESCE(ask_volume3, 0) / 3.0
                + COALESCE(ask_volume4, 0) * 0.25
                + COALESCE(ask_volume5, 0) * 0.2
            ) AS ask_weighted_volume,
            (
                bid_price1 * COALESCE(bid_volume1, 0) * 1.0
                + bid_price2 * COALESCE(bid_volume2, 0) * 0.5
                + bid_price3 * COALESCE(bid_volume3, 0) / 3.0
                + bid_price4 * COALESCE(bid_volume4, 0) * 0.25
                + bid_price5 * COALESCE(bid_volume5, 0) * 0.2
            )
            / NULLIF(
                COALESCE(bid_volume1, 0) * 1.0
                + COALESCE(bid_volume2, 0) * 0.5
                + COALESCE(bid_volume3, 0) / 3.0
                + COALESCE(bid_volume4, 0) * 0.25
                + COALESCE(bid_volume5, 0) * 0.2,
                0
            ) AS weighted_bid_cost,
            (
                ask_price1 * COALESCE(ask_volume1, 0) * 1.0
                + ask_price2 * COALESCE(ask_volume2, 0) * 0.5
                + ask_price3 * COALESCE(ask_volume3, 0) / 3.0
                + ask_price4 * COALESCE(ask_volume4, 0) * 0.25
                + ask_price5 * COALESCE(ask_volume5, 0) * 0.2
            )
            / NULLIF(
                COALESCE(ask_volume1, 0) * 1.0
                + COALESCE(ask_volume2, 0) * 0.5
                + COALESCE(ask_volume3, 0) / 3.0
                + COALESCE(ask_volume4, 0) * 0.25
                + COALESCE(ask_volume5, 0) * 0.2,
                0
            ) AS weighted_ask_cost
        FROM cte_base
        WHERE cnt >= 20
    ),
    cte_feature AS (
        SELECT
            trading_day,
            instrument,
            rn,
            cnt,
            close,
            pre_close,
            CASE
                WHEN mid_price > 0 AND weighted_bid_cost > 0
                THEN (mid_price - weighted_bid_cost) / mid_price
                ELSE NULL
            END AS bid_gap,
            CASE
                WHEN mid_price > 0 AND weighted_ask_cost > 0
                THEN (weighted_ask_cost - mid_price) / mid_price
                ELSE NULL
            END AS ask_gap,
            CASE
                WHEN mid_price > 0
                     AND weighted_bid_cost > 0
                     AND weighted_ask_cost > 0
                THEN ((weighted_bid_cost + weighted_ask_cost) / 2.0 - mid_price) / mid_price
                ELSE NULL
            END AS book_center,
            CASE
                WHEN mid_price > 0
                THEN (ask_price1 - bid_price1) / mid_price
                ELSE NULL
            END AS spread_ratio,
            (
                bid_weighted_volume - ask_weighted_volume
            ) / NULLIF(
                bid_weighted_volume + ask_weighted_volume,
                0
            ) AS depth_imbalance
        FROM cte_snapshot
        WHERE mid_price > 0
    ),
    cte_daily AS (
        SELECT
            trading_day,
            instrument,
            LAST(close ORDER BY rn) AS last_close,
            FIRST(pre_close ORDER BY rn) AS pre_close,
            AVG(book_center) AS book_center,
            AVG(spread_ratio) AS spread_ratio,
            AVG(CASE WHEN rn > cnt * 2 / 3 THEN depth_imbalance END) AS late_depth_imbalance
        FROM cte_feature
        GROUP BY instrument, trading_day
    ),
    cte_roll AS (
        SELECT
            trading_day,
            instrument,
            last_close / NULLIF(pre_close, 0) - 1.0 AS daily_ret,
            book_center,
            spread_ratio,
            late_depth_imbalance,
            AVG(book_center) OVER (
                PARTITION BY instrument
                ORDER BY trading_day
                ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
            ) AS prev3_book_center,
            AVG(last_close / NULLIF(pre_close, 0) - 1.0) OVER (
                PARTITION BY instrument
                ORDER BY trading_day
                ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
            ) AS prev5_ret_trend,
            AVG(spread_ratio) OVER (
                PARTITION BY instrument
                ORDER BY trading_day
                ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
            ) AS prev3_spread_ratio
        FROM cte_daily
    ),
    cte_factor AS (
        SELECT
            trading_day,
            instrument,
            COALESCE(prev3_book_center, 0) - COALESCE(book_center, 0) AS book_center_down,
            CASE
                WHEN prev5_ret_trend < 0
                THEN daily_ret - prev5_ret_trend
                ELSE 0
            END AS decline_narrow,
            COALESCE(prev3_spread_ratio, 0) - COALESCE(spread_ratio, 0) AS spread_tighten,
            CASE
                WHEN (1.0 + COALESCE(late_depth_imbalance, 0)) / 2.0 < 0 THEN 0
                WHEN (1.0 + COALESCE(late_depth_imbalance, 0)) / 2.0 > 1 THEN 1
                ELSE (1.0 + COALESCE(late_depth_imbalance, 0)) / 2.0
            END AS depth_confirm
        FROM cte_roll
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        (
            CASE
                WHEN book_center_down > 0 AND decline_narrow > 0
                THEN book_center_down * decline_narrow
                ELSE 0
            END
            * (1.0 + 0.5 * CASE WHEN spread_tighten > 0 THEN spread_tighten ELSE 0 END)
            * depth_confirm
        ) AS factor
    FROM cte_factor
    ORDER BY date, instrument
    """

    df = dai.query(
        sql,
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()

    df = clean_factor(df)
    df = df[
        (df["date"] >= eval_start.normalize())
        & (df["date"] <= pd.to_datetime(end_date).normalize())
    ].copy()

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()

    # 缺少有效盘口结构时输出中性值 0，避免覆盖率失败；不跨日填充。
    df = pd.merge(df, stk_pool, how="right", on=["date", "instrument"])
    df["factor"] = df["factor"].fillna(0)
    return clean_factor(df)


def clean_factor(df):
    import numpy as np
    import pandas as pd

    df = df[["date", "instrument", "factor"]].copy()
    df["date"] = pd.to_datetime(df["date"])
    df["factor"] = pd.to_numeric(df["factor"], errors="coerce")
    df["factor"] = df["factor"].replace([np.inf, -np.inf], np.nan)
    df = df.drop_duplicates(["date", "instrument"], keep="last")
    df = df.sort_values(["date", "instrument"]).reset_index(drop=True)
    return df


def run_factor_by_trading_day(datasources, start_date, end_date):
    """
    平台自测辅助函数：按交易日逐日计算完整训练期，降低单次分钟查询内存压力，
    同时让每个交易日的时间边界最容易审计。
    """
    import pandas as pd
    import dai

    trading_dates = dai.query(
        """
        SELECT DISTINCT date
        FROM bigalpha_2026_instruments
        ORDER BY date
        """,
        filters={"date": [start_date, end_date]},
    ).df()["date"]

    frames = []
    total = len(trading_dates)

    for i, dt in enumerate(trading_dates, start=1):
        day = pd.to_datetime(dt).strftime("%Y-%m-%d")
        day_start = f"{day} 00:00:00"
        day_end = f"{day} 23:59:59"
        print(f"[order_book_cost_trend_absorption_v3] {i}/{total} {day}", flush=True)

        day_df = main(datasources, day_start, day_end)
        if len(day_df) > 0:
            frames.append(day_df)

    if not frames:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    return clean_factor(pd.concat(frames, ignore_index=True))


if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    start_date = "2019-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    logger.info(f"按交易日逐日计算因子，区间：{start_date} ~ {end_date}")
    factor_data = run_factor_by_trading_day(datasources, start_date, end_date)

    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-06-27 12:36:41] [info     ] 按交易日逐日计算因子，区间：2019-01-01 00:00:00 ~ 2024-12-31 23:59:59
[order_book_cost_trend_absorption_v3] 1/1456 2019-01-02
[order_book_cost_trend_absorption_v3] 2/1456 2019-01-03
[order_book_cost_trend_absorption_v3] 3/1456 2019-01-04
[order_book_cost_trend_absorption_v3] 4/1456 2019-01-07
[order_book_cost_trend_absorption_v3] 5/1456 2019-01-08
[order_book_cost_trend_absorption_v3] 6/1456 2019-01-09
[order_book_cost_trend_absorption_v3] 7/1456 2019-01-10
[order_book_cost_trend_absorption_v3] 8/1456 2019-01-11
[order_book_cost_trend_absorption_v3] 9/1456 2019-01-14
[order_book_cost_trend_absorption_v3] 10/1456 2019-01-15
[order_book_cost_trend_absorption_v3] 11/1456 2019-01-16
[order_book_cost_trend_absorption_v3] 12/1456 2019-01-17
[order_book_cost_trend_absorption_v3] 13/1456 2019-01-18
[order_book_cost_trend_absorption_v3] 14/1456 2019-01-21
[order_book_cost_trend_absorption_v3] 15/1456 2019-01-22
[order_book_cost_trend_absorption_v3] 16/1456 2019-01-23
[order_

: 